# Climate Experiments Notebook

The purpose of this notebook is to combine efforts from data prep notebooks and run a full climate experiment as per the scope of our research project. Integrated gradients are used to incorporate explainable AI components and maps are generated for clear interpretability and visualization.

### Imports and File Stitching

In [1]:
# Machine learning imports 
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            # Tell TF to only take what it needs, not everything at once
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

from tensorflow.keras import models, layers
from tensorflow.keras import backend as K

# General Imports
import pandas as pd
import numpy as np
import os
import sys

# Modeling 
from sklearn.model_selection import train_test_split

# ----File Stitching----
# If in climate_experiments folder, cd back to MamalakisResearch folder
if os.path.basename(os.getcwd()) == "climate_experiments":
    os.chdir('..')
# If a file is in /data_prep_viz/prep/, access it by telling the system to look at that path as well as current path
sys.path.append(os.path.join(os.getcwd(), '..', 'data_prep_viz/prep'))

In [2]:
import tensorflow as tf
print(tf.__version__) 
print(tf.config.list_physical_devices('GPU'))

2.10.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
%%capture
%run "data_prep_viz/prep/get_cnn_tensors.ipynb" 

KeyboardInterrupt: 

KeyboardInterrupt: 

### Compute Integrated Gradients

In [ ]:
def get_gradients(inputs, model, top_pred_idx=None):
    """Computes the gradients of outputs w.r.t input image.

    Args:
        inputs: 2D/3D/4D matrix of samples
        top_pred_idx: (optional) Predicted label for the x_data
                      if classification problem. If regression,
                      do not include.

    Returns:
        Gradients of the predictions w.r.t img_input
    """
    inputs = tf.cast(inputs, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(inputs)
        
        # Run the forward pass of the layer and record operations
        # on GradientTape.
        preds = model(inputs, training=False)  
        
        # For classification, grab the top class
        if top_pred_idx is not None:
            preds = preds[:, top_pred_idx]
        
    # Use the gradient tape to automatically retrieve
    # the gradients of the trainable variables with respect to the loss.        
    grads = tape.gradient(preds, inputs)
    return grads

In [ ]:
def get_integrated_gradients(inputs, model, baseline=None, num_steps=50, top_pred_idx=None):
    # 1. Ensure inputs and baseline are float32
    inputs = inputs.astype(np.float32)
    
    if baseline is None:
        # Fallback to zeros if no baseline provided
        baseline = np.zeros_like(inputs).astype(np.float32)
    else:
        baseline = baseline.astype(np.float32)
        # Ensure baseline has a leading dimension if it's a single mean map
        if baseline.ndim == inputs.ndim - 1:
            baseline = np.expand_dims(baseline, axis=0)

    # 2. Generate interpolation steps
    # We use np.linspace to create the scaling factors (alphas)
    alphas = np.linspace(0.0, 1.0, num_steps + 1)
    
    # 3. Compute Gradients along the path
    # We iterate through the interpolation path from baseline to input
    all_grads = []
    for alpha in alphas:
        # Interpolate: baseline + alpha * (input - baseline)
        step_input = baseline + alpha * (inputs - baseline)
        
        # Get gradients for this specific step
        grad = get_gradients(step_input, model, top_pred_idx=top_pred_idx)
        all_grads.append(grad)
    
    # 4. Convert to tensor for averaging
    # Shape: (num_steps + 1, batch, vars, lat, lon)
    all_grads = tf.convert_to_tensor(all_grads, dtype=tf.float32)

    # 5. Approximate the integral (Trapezoidal Rule)
    # Average the gradients of adjacent steps
    grads_at_step_ends = (all_grads[:-1] + all_grads[1:]) / 2.0
    avg_grads = tf.reduce_mean(grads_at_step_ends, axis=0)

    # 6. Final IG calculation: (input - baseline) * average gradient
    integrated_grads = (inputs - baseline) * avg_grads.numpy()
    
    return integrated_grads

In [ ]:
# sophie edit: said i didnt have some of the libs (feel like something didnt connect with the git but im scared)
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
import netCDF4 as nc
import pandas as pd

### Train the CNN

In [ ]:
def cnn_training(X_data, y_data, learning_rate=0.0001, epochs=200, batch_size=64):
    # prep indices
    n_samples = X_data.shape[0]
    indices = np.arange(n_samples) # [0, 1, 2, ..., N-1]

    X = np.transpose(X_data, (0, 2, 3, 1))  # (N, lat, lon, 7)
    y = y_data.astype(np.float32)           # (N, 1)
    
    # split test set (50 samples) 
        # passing indices to keep track of the indices that are goin in the set 
    X_rem, X_test, y_rem, y_test, idx_rem, test_indices = train_test_split(
        X, y, indices,
        test_size=50,
        stratify=y
    )

    # val split (from remaning 450 samples)
    X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
        X_rem, y_rem, idx_rem,
        test_size=50,
        stratify=y_rem
    )
    
    lat, lon = X_train.shape[1], X_train.shape[2]
    model = models.Sequential([
        layers.Input(shape=(lat, lon, 7)),
        
        #  CNN block (64 filters) with two convs, then pool
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        # CNN block 32 kernels (conv + pool)
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        # CNN block 16 kernels (conv only)
        layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(8, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(8, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),             
        layers.Dense(50, activation="relu"),
        layers.Dense(10, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=2
    )

    # Take out for now 
    # checkpoint = tf.keras.callbacks.ModelCheckpoint(
    #     filepath='best_model.h5',
    #     monitor='val_loss',
    #     save_best_only=True
    # )

    # train model 
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Return everything needed for the large loop
    return model, X_train, y_train, X_test, y_test, test_indices

In [ ]:
# Reimport in case of error 
import xarray as xr

def run_climate_experiment(scenarios, early_starts, model_list, data_path):

    # initializing list to store results from each early/late period iteration
    all_results = []
    
    # looping thru each scenario (ssp119, ssp126)
    for scenario in scenarios:
        # for every early start year in early_starts list
        for early_start in early_starts:
            # make late period start years as 10 plus the early start year
            # Go to 2105 since non-inclusive, otherwise would skip 2095 as a late start
            late_starts = np.arange(early_start + 10, 2105, 10) 
            
            # Had early start in here too while zipped, maybe we can replace double for loop with that?
            for late_start in late_starts:

                if late_start - early_start == 10:
                    
                    print(f"Processing: {scenario} | Early: {early_start} | Late: {late_start}")
                    
                    # prepping data for every early and late 10yr time period combo 
                    X_data, y_data = get_cnn_tensors(
                        model_list, scenario, data_path, 
                        st_early=early_start, end_early=early_start+9, 
                        st_late=late_start, end_late=late_start+9
                    )
                    
                    # training data 
                    model, X_train, y_train, X_test, y_test, test_idx = cnn_training(X_data, y_data)
                    
                    # predicting in batches of 32 
                    preds = model.predict(X_test, batch_size=32).flatten()
                    
                    # --- CALCULATE ACCURACY ---
                    # Convert probabilities to binary 0 or 1 using 0.5 as threshold
                    # Caroline fix: Adding 0.5 itself to the 1 category
                    binary_preds = (preds >= 0.5).astype(int)
                    # Compare to y_test (flattened to match shapes)
                    accuracy = np.mean(binary_preds == y_test.flatten())
                    
                    print(f"--> Iteration Accuracy: {accuracy:.2%}")
                    
                    # XAI STUFF: 
                    # baseline is the mean of early period from training set
                    early_idx = np.where(y_train == 0)[0]
                    baseline = np.mean(X_train[early_idx], axis=0, keepdims=True)
                    
                    # getting late indices for X_test set 
                    late_test_idx = np.where(y_test == 1)[0]
                    ig_samples = X_test[late_test_idx]
                    
                    # integrated gradient calculation based on the early period baseline on the late period stuff 
                    ig_output = get_integrated_gradients(ig_samples, model, baseline)
                    if hasattr(ig_output, 'numpy'): 
                        ig_output = ig_output.numpy()

                    late_test_idx = np.where(y_test == 1)[0]
                    y_test_filtered = y_test[late_test_idx].flatten() # Convert (25, 1) to (25,)
                    preds_filtered = preds[late_test_idx]

                    # Update Sophie made to saving logic
                    nc_filename = f"results_batches/res_{scenario}_{early_start}_{late_start}.nc"
                
                    # Pass filtered data of 25 samples (not whole test set of 50)
                    save_iteration_netcdf(ig_output, y_test_filtered, preds_filtered, 
                                        scenario, early_start, late_start, nc_filename)
                    
                    final_accuracy = np.mean((preds >= 0.5).astype(int) == y_test.flatten())
                    print(f"final accuracy: {final_accuracy: .2%}")
                    
                    # Keep the summary of all 50 samples for CSV (should we tho?)
                    summary_row = pd.DataFrame([{
                        'scenario': scenario,
                        'early_yr': early_start,
                        'late_yr': late_start,
                        'mean_pred': np.mean(preds), 
                        'accuracy': final_accuracy 
                    }])
                    summary_row.to_csv("experiment_summary.csv", mode='a', 
                                    header=not os.path.exists("experiment_summary.csv"), 
                                    index=False)
                    
                    del ig_output, X_data, y_data, X_train, y_train, X_test

    # Optional return (all on hard drive) I think we can delete this line?
    return None

    K.clear_session()


# Another import in case of error
import xarray as xr

# Helper function
def save_iteration_netcdf(ig_data, y_true, y_pred, scenario, early, late, filename):
    """
    Saves a single iteration's spatial heatmaps to NetCDF.
    Squeezes 4D tensors to 3D to ensure Xarray dimension compatibility.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    
    ds = xr.Dataset(
        data_vars={
            "ig_heatmaps": (("sample", "lat", "lon", "feature"), ig_data),
            "y_true": (("sample",), y_true),
            "y_pred": (("sample",), y_pred)
        },
        coords={
            "scenario": scenario,
            "early_yr": early,
            "late_yr": late
        }
    )
    ds.to_netcdf(filename)

In [ ]:
# Are we doing anything w NetCDF here, can we get rid of those parts?
def save_results(results_list, filename="experiment_results.nc"):
    """
    A function to save nested results into a CSV
    """
    
    summary_df = pd.DataFrame([{
        'scenario': r['scenario'],
        'early': r['early_yr'],
        'late': r['late_yr'],
        'mean_pred': np.mean(r['y_pred'])
    } for r in results_list])
    summary_df.to_csv("experiment_summary.csv", index=False)
    
    print("Results saved to experiment_summary.csv and (optionally) NetCDF.")


In [ ]:
results = run_climate_experiment(['ssp119', 'ssp126'], [2015, 2025, 2035, 2045, 2055, 2065, 2075, 2085, 2095], model_list, data_path) 
# Run this 6+ times to generate and quantify uncertainty

Processing: ssp119 | Early: 2015 | Late: 2025
processing model: CNRM_ESM2-1


C:\Users\student\AppData\Local\Temp\ipykernel_4688\1044987437.py:94: RuntimeWarning: Mean of empty slice
  p_mean = np.nanmean(baseline_slice, axis=(0, 2))
C:\Users\student\AppData\Local\Temp\ipykernel_4688\1044987437.py:118: RuntimeWarning: Mean of empty slice
  yearly_val = calc_func(annual_slice, axis=1)


processing model: MIROC6
processing model: MPI-ESM1-2-LR
processing model: MRI-ESM2-0
processing model: UKESM1-0-LL
Epoch 1/200
7/7 [==============================] - 3s 162ms/step - loss: 0.6904 - accuracy: 0.5000 - val_loss: 0.6869 - val_accuracy: 0.5000
Epoch 2/200
7/7 [==============================] - 1s 112ms/step - loss: 0.6866 - accuracy: 0.5000 - val_loss: 0.6834 - val_accuracy: 0.5000
Epoch 3/200
7/7 [==============================] - 1s 112ms/step - loss: 0.6833 - accuracy: 0.5000 - val_loss: 0.6800 - val_accuracy: 0.5000
Epoch 4/200
7/7 [==============================] - 1s 112ms/step - loss: 0.6797 - accuracy: 0.5000 - val_loss: 0.6770 - val_accuracy: 0.5000
Epoch 5/200
7/7 [==============================] - 1s 111ms/step - loss: 0.6757 - accuracy: 0.5000 - val_loss: 0.6739 - val_accuracy: 0.5000
Epoch 6/200
7/7 [==============================] - 1s 111ms/step - loss: 0.6708 - accuracy: 0.5000 - val_loss: 0.6700 - val_accuracy: 0.5000
Epoch 7/200
7/7 [=====================

C:\Users\student\AppData\Local\Temp\ipykernel_4688\1044987437.py:94: RuntimeWarning: Mean of empty slice
  p_mean = np.nanmean(baseline_slice, axis=(0, 2))
c:\Users\student\.conda\envs\climate\lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\student\AppData\Local\Temp\ipykernel_4688\1044987437.py:118: RuntimeWarning: Mean of empty slice
  yearly_val = calc_func(annual_slice, axis=1)


processing model: MIROC6
processing model: MPI-ESM1-2-LR
processing model: MRI-ESM2-0
processing model: UKESM1-0-LL
Epoch 1/200
7/7 [==============================] - 2s 140ms/step - loss: 0.6917 - accuracy: 0.4725 - val_loss: 0.6866 - val_accuracy: 0.6000
Epoch 2/200
7/7 [==============================] - 1s 111ms/step - loss: 0.6900 - accuracy: 0.5700 - val_loss: 0.6835 - val_accuracy: 0.6800
Epoch 3/200
7/7 [==============================] - 1s 112ms/step - loss: 0.6881 - accuracy: 0.6500 - val_loss: 0.6795 - val_accuracy: 0.7400
Epoch 4/200
7/7 [==============================] - 1s 111ms/step - loss: 0.6860 - accuracy: 0.6925 - val_loss: 0.6742 - val_accuracy: 0.7400
Epoch 5/200
7/7 [==============================] - 1s 112ms/step - loss: 0.6825 - accuracy: 0.6625 - val_loss: 0.6665 - val_accuracy: 0.7400
Epoch 6/200
7/7 [==============================] - 1s 119ms/step - loss: 0.6776 - accuracy: 0.6500 - val_loss: 0.6547 - val_accuracy: 0.7000
Epoch 7/200
7/7 [=====================

...

C:\Users\student\AppData\Local\Temp\ipykernel_4688\1044987437.py:94: RuntimeWarning: Mean of empty slice
  p_mean = np.nanmean(baseline_slice, axis=(0, 2))
c:\Users\student\.conda\envs\climate\lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\student\AppData\Local\Temp\ipykernel_4688\1044987437.py:118: RuntimeWarning: Mean of empty slice
  yearly_val = calc_func(annual_slice, axis=1)


processing model: MIROC6
processing model: MPI-ESM1-2-LR
processing model: MRI-ESM2-0
processing model: UKESM1-0-LL
Epoch 1/200
7/7 [==============================] - 2s 145ms/step - loss: 0.6924 - accuracy: 0.5000 - val_loss: 0.6925 - val_accuracy: 0.5000
Epoch 2/200
7/7 [==============================] - 1s 117ms/step - loss: 0.6915 - accuracy: 0.5000 - val_loss: 0.6920 - val_accuracy: 0.5000
Epoch 3/200
7/7 [==============================] - 1s 119ms/step - loss: 0.6905 - accuracy: 0.5000 - val_loss: 0.6916 - val_accuracy: 0.5000
Epoch 4/200
7/7 [==============================] - 1s 118ms/step - loss: 0.6892 - accuracy: 0.5000 - val_loss: 0.6908 - val_accuracy: 0.5000
Epoch 5/200
7/7 [==============================] - 1s 116ms/step - loss: 0.6873 - accuracy: 0.5000 - val_loss: 0.6897 - val_accuracy: 0.5000
Epoch 6/200
7/7 [==============================] - 1s 116ms/step - loss: 0.6843 - accuracy: 0.5000 - val_loss: 0.6885 - val_accuracy: 0.5000
Epoch 7/200
7/7 [=====================

C:\Users\student\AppData\Local\Temp\ipykernel_4688\1044987437.py:94: RuntimeWarning: Mean of empty slice
  p_mean = np.nanmean(baseline_slice, axis=(0, 2))
c:\Users\student\.conda\envs\climate\lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\student\AppData\Local\Temp\ipykernel_4688\1044987437.py:118: RuntimeWarning: Mean of empty slice
  yearly_val = calc_func(annual_slice, axis=1)


processing model: MIROC6
processing model: MPI-ESM1-2-LR
processing model: MRI-ESM2-0
processing model: UKESM1-0-LL
Epoch 1/200
7/7 [==============================] - 2s 146ms/step - loss: 0.6942 - accuracy: 0.4575 - val_loss: 0.6934 - val_accuracy: 0.4000
Epoch 2/200
7/7 [==============================] - 1s 124ms/step - loss: 0.6925 - accuracy: 0.5700 - val_loss: 0.6927 - val_accuracy: 0.4600
Epoch 3/200
7/7 [==============================] - 1s 118ms/step - loss: 0.6914 - accuracy: 0.6275 - val_loss: 0.6918 - val_accuracy: 0.5400
Epoch 4/200
7/7 [==============================] - 1s 117ms/step - loss: 0.6901 - accuracy: 0.6425 - val_loss: 0.6910 - val_accuracy: 0.5600
Epoch 5/200
7/7 [==============================] - 1s 118ms/step - loss: 0.6883 - accuracy: 0.6750 - val_loss: 0.6903 - val_accuracy: 0.5800
Epoch 6/200
7/7 [==============================] - 1s 120ms/step - loss: 0.6859 - accuracy: 0.6775 - val_loss: 0.6883 - val_accuracy: 0.5600
Epoch 7/200
7/7 [=====================

KeyboardInterrupt: 